Intro and references

In [ ]:
!nvidia-smi

Sat May  2 08:09:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import glob
import json
import os
import numpy as np
import pandas as pd
import random
import rdkit
import shutil
import subprocess
import sys
import tarfile
import torch
import os

from collections import Counter
from rdkit.Chem.Draw import mplCanvas

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Google Colab environment paths
    ROOT_DIR = "/content"
    # If you need to mount Google Drive, uncomment the following:
    from google.colab import drive
    drive.mount('/content/drive')
    SCRIPTS_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/unibo_bigdata/scripts")
    print("Detected Environment: Google Colab")
else:
    # Local environment paths
    ROOT_DIR = os.getcwd()
    SCRIPTS_DIR = os.path.join(ROOT_DIR, "/scripts")
    print(f"Detected Environment: Local ({sys.platform})")

print(f"Root Directory: {ROOT_DIR}")
print(f"Data Directory: {SCRIPTS_DIR}")

# Prepare the data and load it all from GBucket

We save to GDrive (if was not already done). Because all CAFA-5 entries are from UniProt and usually have PDB in AlphaDB.

In [ ]:
!pip -q install -U crcmod

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/gcs_key.json"

In [ ]:
!gcloud auth activate-service-account --key-file=/content/gcs_key.json
!gcloud config set project lithe-sunset-283907

Activated service account credentials for: [stor-9@lithe-sunset-283907.iam.gserviceaccount.com]


To take a quick anonymous survey, run:
  $ gcloud survey

Updated property [core/project].


In [ ]:
!mkdir /content/shards

In [ ]:
!gsutil -m cp \
  gs://protein_shards/train_taxonomy.tsv \
  gs://protein_shards/train_terms.tsv \
  gs://protein_shards/VarBatchSampler.py \
  gs://protein_shards/Schake_trained_weights.pt \
  gs://protein_shards/Schake_model_v2.py \
  gs://protein_shards/go-basic.obo \
  gs://protein_shards/IA.txt \
  gs://protein_shards/gearnet_embeds.zip \
  gs://protein_shards/t5_prot_baseline_train_embeds.npy.zip \
  gs://protein_shards/t5_prot_baseline_train_ids.npy.zip \
  /content/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://protein_shards/train_taxonomy.tsv...
Copying gs://protein_shards/train_terms.tsv...
Copying gs://protein_shards/VarBatchSampler.py...
Copying gs://protein_shards/Schake_trained_weights.pt...
Copying gs://protein_shards/Schake_model_v2.py...
Copying gs://protein_shards/go-basic.obo...
Copying gs://protein_shards/IA.txt...
Copying gs://protein_shards/gearnet_embeds.zip...
Copying gs://protein_shards/t5_prot_baseline_train_embeds.npy.zip...
Copying gs://protein_shards/t5_prot_baseline_train_ids.npy.zip...


In [ ]:
!unzip t5_prot_baseline_train_embeds.npy.zip
!unzip t5_prot_baseline_train_ids.npy.zip
!mkdir -p /content/extracted && unzip -q /content/gearnet_embeds.zip -d /content/extracted && mv /content/extracted/content/drive/MyDrive/go_finetune_runs /content/gearnet_embeds && rm -rf /content/extracted

In [ ]:
taxonomy_file_path = '/content/train_taxonomy.tsv'
try:
    taxonomy_df = pd.read_csv(taxonomy_file_path, sep='\t')
    if 'EntryID' in taxonomy_df.columns:
        ACCESSIONS = taxonomy_df['EntryID'].tolist()
        print(f"Loaded {len(ACCESSIONS)} accessions from {taxonomy_file_path}")
        print(f"First 5 accessions: {ACCESSIONS[:5]}")
    else:
        print(f"Error: 'EntryID' column not found in {taxonomy_file_path}")
except FileNotFoundError:
    print(f"Error: File not found at {taxonomy_file_path}. Please ensure it's in your Drive root.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

In [ ]:
train_terms = pd.read_csv("/content/train_terms.tsv",sep="\t")
print(train_terms.shape)

In [ ]:
train_terms

In [ ]:
#this must be done for the first time only - I mean only for .tar shards to pt shards conversion
!gsutil -m cp -r gs://protein_shards/shards/*.tar /content/shards

In [ ]:
#!gsutil -m cp -r gs://protein_shards/pt_shards/*.pt /content/pt_shards

In [ ]:
#!gsutil -m cp -r gs://protein_shards/pt_shards/*.pt /content/pt_shards

In [ ]:
#!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
#!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
#!pip install torch_geometric -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install mdtraj

In [ ]:
!pip install webdataset

In [ ]:
#!mkdir wheels

In [ ]:
!gsutil -m cp -r gs://protein_shards/wheels/* /content/wheels/

In [ ]:
!pip install --no-index --find-links=/content/wheels/torch2.10.0-cu128-py312 \
  torch-scatter torch-cluster
!pip install torch-geometric

In [ ]:
%%bash
set -euxo pipefail

cd /content

# -----------------------------
# micromamba bootstrap
# -----------------------------
if [ ! -x /content/bin/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
fi

export MAMBA_ROOT_PREFIX=/content/micromamba
MICROMAMBA=/content/bin/micromamba
ENV_NAME=torchdrug38

# -----------------------------
# detect GPU generation
# -----------------------------
GPU_NAME="$(nvidia-smi --query-gpu=name --format=csv,noheader | head -n 1 || true)"
echo "Detected GPU: ${GPU_NAME}"

IS_BLACKWELL=0

# Name-based detection first
shopt -s nocasematch
if [[ "${GPU_NAME}" == *"RTX 50"* ]] || \
   [[ "${GPU_NAME}" == *"5090"* ]] || \
   [[ "${GPU_NAME}" == *"5080"* ]] || \
   [[ "${GPU_NAME}" == *"5070"* ]] || \
   [[ "${GPU_NAME}" == *"5060"* ]] || \
   [[ "${GPU_NAME}" == *"5050"* ]] || \
   [[ "${GPU_NAME}" == *"Blackwell"* ]] || \
   [[ "${GPU_NAME}" == *"GB200"* ]] || \
   [[ "${GPU_NAME}" == *"B200"* ]] || \
   [[ "${GPU_NAME}" == *"B100"* ]] || \
   [[ "${GPU_NAME}" == *"RTX PRO 6000 Blackwell"* ]]; then
  IS_BLACKWELL=1
fi
shopt -u nocasematch

echo "IS_BLACKWELL=${IS_BLACKWELL}"

# -----------------------------
# system build tools
# -----------------------------
apt-get update
apt-get install -y build-essential

# -----------------------------
# recreate env
# -----------------------------
rm -rf "/content/micromamba/envs/${ENV_NAME}"

if [ "${IS_BLACKWELL}" = "1" ]; then
  # ---------------------------------
  # Blackwell path
  # Python 3.10 because current stable PyTorch requires 3.10+,
  # and TorchDrug compatibility advertises support through 3.10.
  # ---------------------------------
  $MICROMAMBA create -y -n "${ENV_NAME}" -c conda-forge \
    python=3.10 \
    "numpy<2" \
    python-lmdb \
    cffi \
    pip \
    rdkit \
    pandas \
    tqdm \
    scipy \
    matplotlib \
    ninja
else
  # ---------------------------------
  # Older / pre-Blackwell path
  # Matches your working Torch 1.13.1 + cu117 stack
  # ---------------------------------
  $MICROMAMBA create -y -n "${ENV_NAME}" -c conda-forge \
    python=3.8 \
    python-lmdb \
    cffi \
    pip \
    ninja
fi

PY="/content/micromamba/envs/${ENV_NAME}/bin/python"
PIP="$PY -m pip"

# Put env binaries first on PATH
export PATH="/content/micromamba/envs/${ENV_NAME}/bin:${PATH}"

# -----------------------------
# core packaging tools
# -----------------------------
$PIP install --no-cache-dir -U pip setuptools wheel

# -----------------------------
# install torch / pyg / torchdrug
# -----------------------------
if [ "${IS_BLACKWELL}" = "1" ]; then
  # PyTorch 2.7 + CUDA 12.8 for Blackwell
  $PIP install --no-cache-dir \
    torch==2.7.0 torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu128

  # remove anything mismatched first
  $PIP uninstall -y \
    pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv torch_geometric torchdrug || true

  # matching PyG compiled wheels
  $PIP install --no-cache-dir \
    pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-2.7.0+cu128.html

  $PIP install --no-cache-dir torch_geometric

  # TorchDrug without letting pip "helpfully" rebuild deps
  $PIP install --no-cache-dir --no-deps torchdrug

  # extras commonly needed around ESM-GearNet / TorchDrug
  $PIP install --no-cache-dir --no-deps webdataset
  $PIP install --no-cache-dir decorator networkx jinja2 fair-esm
else
  # PyTorch 1.13.1 + CUDA 11.7 for older GPUs
  $PIP install --no-cache-dir \
    torch==1.13.1+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117

  $PIP uninstall -y \
    pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv torch_geometric || true

  $PIP install --no-cache-dir \
    pyg_lib==0.3.1+pt113cu117 \
    torch_scatter==2.1.1+pt113cu117 \
    torch_sparse==0.6.17+pt113cu117 \
    torch_cluster==1.6.1+pt113cu117 \
    torch_spline_conv==1.2.2+pt113cu117 \
    -f https://data.pyg.org/whl/torch-1.13.1+cu117.html

  $PIP install --no-cache-dir torch_geometric==2.3.1
  $PIP install --no-cache-dir rdkit-pypi torchdrug webdataset pandas tqdm fair-esm
fi

# -----------------------------
# verification
# -----------------------------
MPLBACKEND=Agg $PY - <<'PYCODE'

print("python:", os.sys.version)
print("python_executable:", os.sys.executable)
print("torch:", torch.__version__)
print("torch_cuda:", torch.version.cuda)
print("cuda_available:", torch.cuda.is_available())
print("ninja_on_path:", shutil.which("ninja"))

if torch.cuda.is_available():
    print("gpu_name:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

mods = [
    "lmdb",
    "pyg_lib",
    "torch_scatter",
    "torch_sparse",
    "torch_cluster",
    "torch_spline_conv",
    "torch_geometric",
    "rdkit",
    "torchdrug",
]

for name in mods:
    try:
        mod = __import__(name)
        print(f"{name}: OK", getattr(mod, "__version__", ""))
    except Exception as e:
        print(f"{name}: FAIL {repr(e)}")

try:
    from torchdrug import models
    print("GearNet exists:", hasattr(models, "GearNet"))
except Exception as e:
    print("torchdrug models import failed:", repr(e))
PYCODE

In [ ]:
%%bash
set -euxo pipefail

export MAMBA_ROOT_PREFIX=/content/micromamba
MICROMAMBA=/content/bin/micromamba
PY=/content/micromamba/envs/torchdrug38/bin/python

$MICROMAMBA install -y -n torchdrug38 -c conda-forge "rdkit=2023.09.*" ninja

MPLBACKEND=Agg $PY - <<'PY'
print("rdkit", rdkit.__version__)
print("ninja", shutil.which("ninja"))
print("mplCanvas OK")
print("torchdrug OK", torchdrug.__version__)
PY

In [ ]:
!/content/bin/micromamba install -y -p /content/micromamba/envs/torchdrug38 h5py
!/content/micromamba/envs/torchdrug38/bin/python -m pip install atom3d
!git clone https://github.com/DeepGraphLearning/ESM-GearNet.git


In [ ]:
%%bash
set -euxo pipefail

curl https://sh.rustup.rs -sSf | sh -s -- -y
export PATH="$HOME/.cargo/bin:$PATH"

/content/micromamba/envs/torchdrug38/bin/python -m pip install --no-cache-dir \
  "tokenizers==0.10.3" "transformers==4.14.1"

In [ ]:
# !/content/micromamba/envs/torchdrug38/bin/python -m pip install --no-cache-dir transformers

In [ ]:
# !/content/micromamba/envs/torchdrug38/bin/python -m pip install "transformers==4.14.1"

In [ ]:
# !/content/micromamba/envs/torchdrug38/bin/python -m pip install "transformers==4.14.1" "tokenizers==0.10.3"

# Prepare CAFA evaluator and helper functions

# Put here my AlphaDB sharding and download code (plus why not fold myself explanations)

#  A strong baseline T5-prot-embeddings

In [ ]:
train_protein_ids = np.load('train_ids.npy')
print(train_protein_ids.shape)

(142246,)


In [ ]:
train_protein_ids

array(['P20536', 'O73864', 'O95231', ..., 'Q5RGB0', 'A0A2R8QMZ5',
       'A0A8I6GHU0'], dtype='<U10')

In [ ]:
train_embeds = np.load('train_embeds.npy')
print(train_embeds.shape)

(142246, 1024)


In [ ]:
train_embeds

array([[ 0.04948843, -0.03293516,  0.03247323, ..., -0.04353154,
         0.0964628 ,  0.07306959],
       [-0.04461636,  0.06492499, -0.08026284, ...,  0.02672353,
         0.02787905, -0.04842958],
       [-0.02012804, -0.04977943,  0.00789446, ..., -0.03610279,
         0.00769301,  0.10623412],
       ...,
       [ 0.01691809,  0.04133058,  0.00079253, ...,  0.0088079 ,
         0.00648063, -0.01334958],
       [ 0.06125151,  0.08340203,  0.0440247 , ...,  0.00138361,
        -0.04754627,  0.01012351],
       [ 0.02160021,  0.06516985,  0.07492343, ...,  0.0496657 ,
        -0.01987522,  0.04471432]])

In [ ]:
!python /content/train_go_mlp_cafa_approx.py \
  --train_terms_tsv /content/train_terms.tsv \
  --train_taxonomy_tsv /content/train_taxonomy.tsv \
  --train_ids_npy /content/train_ids.npy \
  --train_embeds_npy /content/train_embeds.npy \
  --output_dir /content/go_embed_out \
  --num_labels 1500 \
  --epochs 60 \
  --batch_size 5120

2026-04-02 16:17:58.158769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775146678.180859   28025 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775146678.188231   28025 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775146678.206036   28025 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775146678.206074   28025 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775146678.206076   28025 computation_placer.cc:177] computation placer alr

In [ ]:
FINAL TEST
                ns     fmax    wfmax  best_tau
biological_process 0.430142 0.383805      0.11
molecular_function 0.648200 0.583704      0.18
cellular_component 0.689435 0.572120      0.21
mean_Fmax : 0.589259
mean_WFmax: 0.51321
Saved outputs to: /content/out_bilinear_fusion

In [ ]:
FINAL TEST
                ns     fmax    wfmax  best_tau
biological_process 0.412210 0.363421      0.12
molecular_function 0.613728 0.547499      0.14
cellular_component 0.676306 0.553308      0.21
mean_Fmax : 0.567415
mean_WFmax: 0.488076

WITH ULTRA DEEP GUY - WHO COULD TRAIN MORE it was 30 epochs


NOW I WANT TO SEE 60 epochs with ULTRA DEEP GUY WERE GREAT (AND MAYBE SEE DEEPER)
FINAL TEST
                ns     fmax    wfmax  best_tau
biological_process 0.441026 0.395954      0.12
molecular_function 0.644618 0.581341      0.15
cellular_component 0.688027 0.578803      0.21
mean_Fmax : 0.591224
mean_WFmax: 0.518699

NOW I TRY SHALLOW GUY

TRY 3 HEADS DESIGN TOO


In [ ]:
# For GearNet only on one shard-00
FINAL TEST
                ns     fmax    wfmax  best_tau
biological_process 0.268670 0.225313      0.08
molecular_function 0.481261 0.380543      0.21
cellular_component 0.571503 0.342289      0.14
mean_Fmax : 0.440478
mean_WFmax: 0.316048
Saved outputs to: /content/go_mlp_from_saved_embeds
loss was 0.08 on validation

# t5 on one shard gave
FINAL TEST
                ns     fmax    wfmax  best_tau
biological_process 0.261825 0.223745      0.08
molecular_function 0.488283 0.392558      0.20
cellular_component 0.570406 0.345316      0.13
mean_Fmax : 0.440171
mean_WFmax: 0.320539
loss 0.07 on validation


#Best t5 on all shards gave
                ns     fmax    wfmax  best_tau
biological_process 0.441026 0.395954      0.12
molecular_function 0.644618 0.581341      0.15
cellular_component 0.688027 0.578803      0.21
mean_Fmax : 0.591224
mean_WFmax: 0.518699
loss 0.03 on valid

#GEarn net embeds ultra deep guy on all shards
loss 0.054 on valid
FINAL TEST
                ns     fmax    wfmax  best_tau
biological_process 0.396089 0.349349      0.09
molecular_function 0.628175 0.562485      0.15
cellular_component 0.662112 0.535092      0.16
mean_Fmax : 0.562125
mean_WFmax: 0.482309
Saved outputs to: /content/go_mlp_from_saved_embeds_gearnet_all


#OR 100 epochs
FINAL TEST
                ns     fmax    wfmax  best_tau
biological_process 0.409207 0.364938      0.10
molecular_function 0.638563 0.578775      0.15
cellular_component 0.663726 0.545321      0.16
mean_Fmax : 0.570499
mean_WFmax: 0.496344
Saved outputs to: /content/go_mlp_from_saved_embeds_gearnet_all

#An ensemble how can it be worse
FINAL TEST
                ns     fmax    wfmax  best_tau
biological_process 0.430142 0.383805      0.11
molecular_function 0.648200 0.583704      0.18
cellular_component 0.689435 0.572120      0.21

# Download and check GearNet checkpoint

In [ ]:
TAR_FILE = '/content/shards/shard-00.tar'
EXTRACT_DIR = "/content/pdb_scratch/shard0"
os.makedirs(EXTRACT_DIR, exist_ok=True)

print(f"Extracting {TAR_FILE} to {EXTRACT_DIR}")
with tarfile.open(TAR_FILE, "r") as tar:
    members = [m for m in tar if m.isfile() and m.name.endswith(".pdb")]
    tar.extractall(EXTRACT_DIR, members=members)

pdb_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, "**", "*.pdb"), recursive=True))
print("Number of extracted PDB files:", len(pdb_files))
if pdb_files:
    print("First few PDB files:", pdb_files[:3])
else:
    print("No PDB files were extracted.")

Extracting /content/shards/shard-00.tar to /content/pdb_scratch/shard0


/tmp/ipykernel_5231/462313811.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(EXTRACT_DIR, members=members)


Number of extracted PDB files: 6416
First few PDB files: ['/content/pdb_scratch/shard0/AF-0000000000249450.pdb', '/content/pdb_scratch/shard0/AF-0000000000261410.pdb', '/content/pdb_scratch/shard0/AF-0000000065714493.pdb']


In [ ]:
!rm -rf pdb_scratch/shard0

In [ ]:
env = os.environ.copy()

# critical: where TorchDrug caches compiled extension
env["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"

# force a safe non-inline matplotlib backend for torchdrug imports
env["MPLBACKEND"] = "Agg"

# optional but good: ensure correct PATH (ninja lives here)
env["PATH"] = "/content/micromamba/envs/torchdrug_bw/bin:" + env["PATH"]

cmd = [
    "/content/micromamba/envs/torchdrug_bw/bin/python",
    "-u",
    "/content/test_torchdrug_gearnet_gpu.py",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in p.stdout:
    print(line, end="")

rc = p.wait()
print("return code:", rc)

Using device: cuda
Checkpoint already exists: /content/checkpoints/mc_gearnet_edge.pth
Found 6416 pdb files
First few: ['/content/pdb_scratch/shard0/AF-0000000000249450.pdb', '/content/pdb_scratch/shard0/AF-0000000000261410.pdb', '/content/pdb_scratch/shard0/AF-0000000065714493.pdb']
Dataset size: 6416
Packed raw protein batch_size: 2
PDBs in batch: ['/content/pdb_scratch/shard0/AF-0000000000249450.pdb', '/content/pdb_scratch/shard0/AF-0000000000261410.pdb']
Checkpoint loaded from /content/checkpoints/mc_gearnet_edge.pth
Missing keys: 0
Unexpected keys: 0
/content/micromamba/envs/torchdrug_bw/lib/python3.10/site-packages/torchdrug/layers/geometry/graph.py:187: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:62.)
  y = torch.cross(v

In [ ]:
env = os.environ.copy()

# critical: where TorchDrug caches compiled extension
env["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"

# optional but good: ensure correct PATH (ninja lives here)
env["PATH"] = "/content/micromamba/envs/torchdrug38/bin:" + env["PATH"]

cmd = [
    "/content/micromamba/envs/torchdrug38/bin/python",
    "-u",
    "/content/test_torchdrug_gearnet_gpu.py",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in p.stdout:
    print(line, end="")

rc = p.wait()
print("return code:", rc)

Using device: cuda
--2026-04-03 12:37:08--  https://zenodo.org/record/7593637/files/mc_gearnet_edge.pth
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 188.185.43.153, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/7593637/files/mc_gearnet_edge.pth [following]
--2026-04-03 12:37:08--  https://zenodo.org/records/7593637/files/mc_gearnet_edge.pth
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 80700937 (77M) [application/octet-stream]
Saving to: ‘/content/checkpoints/mc_gearnet_edge.pth’

     0K .......... .......... .......... .......... ..........  0%  148K 8m52s
    50K .......... .......... .......... .......... ..........  0%  321K 6m29s
   100K .......... .......... .......... .......... ..........  0%  121M 4m19s
   150K .......... .......... .......... .......... ..........  0%  174M 3m14s
   200

# Prepare Embeddings from GearNet

In [ ]:
TAR_DIR = "/content/shards"
EXTRACT_ROOT = "/content/pdb_scratch"
os.makedirs(EXTRACT_ROOT, exist_ok=True)

tar_files = sorted(glob.glob(os.path.join(TAR_DIR, "*.tar")))
print(f"Found {len(tar_files)} tar shards")

if not tar_files:
    print("No tar files found.")
else:
    total_extracted = 0
    per_shard_counts = {}

    for tar_path in tar_files:
        shard_name = os.path.splitext(os.path.basename(tar_path))[0]   # shard-00
        extract_dir = os.path.join(EXTRACT_ROOT, shard_name)
        os.makedirs(extract_dir, exist_ok=True)

        print(f"\nExtracting {tar_path} -> {extract_dir}")

        with tarfile.open(tar_path, "r") as tar:
            members = [m for m in tar if m.isfile() and m.name.endswith(".pdb")]
            per_shard_counts[shard_name] = len(members)
            tar.extractall(extract_dir, members=members)

        extracted_now = sorted(glob.glob(os.path.join(extract_dir, "**", "*.pdb"), recursive=True))
        print(f"{shard_name}: expected {per_shard_counts[shard_name]} pdbs, found {len(extracted_now)} after extraction")

        if len(extracted_now) != per_shard_counts[shard_name]:
            print(f"WARNING: mismatch in {shard_name}")

        total_extracted += len(extracted_now)

    all_pdb_files = sorted(glob.glob(os.path.join(EXTRACT_ROOT, "**", "*.pdb"), recursive=True))

    print("\n===== SUMMARY =====")
    print(f"Total shards processed: {len(tar_files)}")
    print(f"Total extracted PDB files: {len(all_pdb_files)}")

    for shard_name in sorted(per_shard_counts):
        print(f"{shard_name}: {per_shard_counts[shard_name]} pdbs")

    if all_pdb_files:
        print("First few PDB files:", all_pdb_files[:3])
    else:
        print("No PDB files were extracted.")

Found 20 tar shards

Extracting /content/shards/shard-00.tar -> /content/pdb_scratch/shard-00


/tmp/ipykernel_935/2868314545.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir, members=members)


shard-00: expected 6416 pdbs, found 6416 after extraction

Extracting /content/shards/shard-01.tar -> /content/pdb_scratch/shard-01
shard-01: expected 6375 pdbs, found 6375 after extraction

Extracting /content/shards/shard-02.tar -> /content/pdb_scratch/shard-02
shard-02: expected 6457 pdbs, found 6457 after extraction

Extracting /content/shards/shard-03.tar -> /content/pdb_scratch/shard-03
shard-03: expected 6431 pdbs, found 6431 after extraction

Extracting /content/shards/shard-04.tar -> /content/pdb_scratch/shard-04
shard-04: expected 6402 pdbs, found 6402 after extraction

Extracting /content/shards/shard-05.tar -> /content/pdb_scratch/shard-05
shard-05: expected 6412 pdbs, found 6412 after extraction

Extracting /content/shards/shard-06.tar -> /content/pdb_scratch/shard-06
shard-06: expected 6366 pdbs, found 6366 after extraction

Extracting /content/shards/shard-07.tar -> /content/pdb_scratch/shard-07
shard-07: expected 6398 pdbs, found 6398 after extraction

Extracting /conte

In [ ]:
random.seed(42)
np.random.seed(42)

GEARNET_DIR = "/content/gearnet_embeds"   # change if needed
OUT_DIR = "/content/ggngo_eval_pack"
os.makedirs(OUT_DIR, exist_ok=True)

train_idx = pd.read_csv(os.path.join(GEARNET_DIR, "train_index.csv"))
valid_idx = pd.read_csv(os.path.join(GEARNET_DIR, "valid_index.csv"))
test_idx  = pd.read_csv(os.path.join(GEARNET_DIR, "test_index.csv"))

print("train:", len(train_idx))
print("valid:", len(valid_idx))
print("test :", len(test_idx))

# small_train = same size as test
n_small_train = len(test_idx)
small_train_idx = train_idx.sample(n=n_small_train, random_state=42).reset_index(drop=True)

print("small_train:", len(small_train_idx))

# save subset index files
small_train_idx.to_csv(os.path.join(OUT_DIR, "small_train_index.csv"), index=False)
valid_idx.to_csv(os.path.join(OUT_DIR, "valid_index.csv"), index=False)
test_idx.to_csv(os.path.join(OUT_DIR, "test_index.csv"), index=False)

train: 99627
valid: 12369
test : 12497
small_train: 12497


In [ ]:
SUBSET_ROOT = os.path.join(OUT_DIR, "pdb_subsets")
os.makedirs(SUBSET_ROOT, exist_ok=True)

def copy_subset_pdbs(df, subset_name, root_out):
    subset_dir = os.path.join(root_out, subset_name)
    os.makedirs(subset_dir, exist_ok=True)

    missing = []
    copied = 0

    for _, row in df.iterrows():
        src = row["pdb_file"]
        acc = str(row["accession"])

        if not os.path.exists(src):
            missing.append((acc, src))
            continue

        dst = os.path.join(subset_dir, os.path.basename(src))
        shutil.copy2(src, dst)
        copied += 1

    print(f"{subset_name}: copied {copied} files, missing {len(missing)}")

    if missing:
        miss_df = pd.DataFrame(missing, columns=["accession", "missing_pdb_file"])
        miss_df.to_csv(os.path.join(root_out, f"{subset_name}_missing.csv"), index=False)

copy_subset_pdbs(small_train_idx, "small_train", SUBSET_ROOT)
copy_subset_pdbs(valid_idx,       "valid",       SUBSET_ROOT)
copy_subset_pdbs(test_idx,        "test",        SUBSET_ROOT)

small_train: copied 12497 files, missing 0
valid: copied 12369 files, missing 0
test: copied 12497 files, missing 0


In [ ]:
def save_accession_list(df, path):
    df["accession"].astype(str).to_csv(path, index=False, header=False)

save_accession_list(small_train_idx, os.path.join(OUT_DIR, "small_train_accessions.txt"))
save_accession_list(valid_idx,       os.path.join(OUT_DIR, "valid_accessions.txt"))
save_accession_list(test_idx,        os.path.join(OUT_DIR, "test_accessions.txt"))

print("Saved accession lists")

Saved accession lists


In [ ]:
manifest = {
    "small_train_size": len(small_train_idx),
    "valid_size": len(valid_idx),
    "test_size": len(test_idx),
    "ggngo_mapping": {
        "train_accessions": "small_train",
        "val_accessions": "test"
    }
}

with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved manifest.json")

Saved manifest.json


In [ ]:
TAR_PATH = os.path.join(os.path.dirname(OUT_DIR), "ggngo_eval_pack.tar")

def tar_filter(tarinfo):
    name = tarinfo.name
    base = os.path.basename(name)

    # skip hidden files and nested tar files
    if base.startswith("."):
        return None
    if base.endswith(".tar") or base.endswith(".tar.gz") or base.endswith(".tgz"):
        return None
    return tarinfo

with tarfile.open(TAR_PATH, "w") as tar:
    tar.add(OUT_DIR, arcname=os.path.basename(OUT_DIR), filter=tar_filter)

print("Wrote:", TAR_PATH)

Wrote: /content/ggngo_eval_pack.tar


In [ ]:
!gsutil -m cp /content/ggngo_eval_pack.tar gs://protein_shards/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file:///content/ggngo_eval_pack.tar [Content-Type=application/x-tar]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of compo

In [ ]:
!gsutil -m ls gs://protein_shards/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://protein_shards/


In [ ]:
!rm -rf pdb_scratch/shard0

In [ ]:
!rm -rf go_finetune_runs

In [ ]:
env = os.environ.copy()

# REQUIRED for TorchDrug JIT extension
env["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"

# force a safe non-inline matplotlib backend
env["MPLBACKEND"] = "Agg"

# ensure micromamba env binaries (ninja etc.) are visible
env["PATH"] = "/content/micromamba/envs/torchdrug_bw/bin:" + env["PATH"]

cmd = [
    "/content/micromamba/envs/torchdrug_bw/bin/python",
    "-u",
    "/content/gearnet_go_finetune_pipeline_i.py",

    "--pdb_root", "/content/pdb_scratch",
    "--train_terms_tsv", "/content/train_terms.tsv",
    "--train_taxonomy_tsv", "/content/train_taxonomy.tsv",
    "--checkpoint", "/content/checkpoints/mc_gearnet_edge.pth",
    "--output_dir", "/content/go_finetune_runs/run1",

    "--batch_size", "16",
    "--num_workers", "40",

    "--bpo_top_labels", "1100",
    "--mfo_top_labels", "450",
    "--cco_top_labels", "300",

    "--save_pt",
    "--save_npy",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in p.stdout:
    print(line, end="")

rc = p.wait()
print("return code:", rc)

Streaming output truncated to the last 5000 lines.
embed:train:  48%|████▊     | 3053/6353 [17:27<19:20,  2.84it/s][08:29:44] Explicit valence for atom # 132 C, 5, is greater than permitted

embed:train:  48%|████▊     | 3074/6353 [17:34<15:47,  3.46it/s][08:29:51] Explicit valence for atom # 14041 O, 3, is greater than permitted

embed:train:  49%|████▊     | 3095/6353 [17:41<19:22,  2.80it/s][08:29:58] Explicit valence for atom # 1070 O, 3, is greater than permitted

embed:train:  49%|████▊     | 3097/6353 [17:41<17:32,  3.09it/s][08:29:58] Explicit valence for atom # 3542 O, 3, is greater than permitted

embed:train:  49%|████▉     | 3102/6353 [17:43<16:28,  3.29it/s][08:30:00] Explicit valence for atom # 3768 O, 3, is greater than permitted

embed:train:  49%|████▉     | 3103/6353 [17:43<17:24,  3.11it/s][08:30:00] Explicit valence for atom # 2483 O, 3, is greater than permitted

embed:train:  49%|████▉     | 3112/6353 [17:46<14:42,  3.67it/s][08:30:02] Explicit valence for atom # 

In [ ]:
!cp -r go_finetune_runs/run1 /content/drive/MyDrive/go_finetune_runs

In [ ]:
env = os.environ.copy()

# REQUIRED for TorchDrug JIT extension
env["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"

# ensure micromamba env binaries (ninja etc.) are visible
env["PATH"] = "/content/micromamba/envs/torchdrug38/bin:" + env["PATH"]

cmd = [
    "/content/micromamba/envs/torchdrug38/bin/python",
    "-u",
    "/content/gearnet_go_finetune_pipeline.py",

    "--pdb_root", "/content/pdb_scratch",
    "--train_terms_tsv", "/content/train_terms.tsv",
    "--train_taxonomy_tsv", "/content/train_taxonomy.tsv",
    "--checkpoint", "/content/checkpoints/mc_gearnet_edge.pth",
    "--output_dir", "/content/go_finetune_runs/run1",

    "--batch_size", "2",
    "--num_workers", "10",

    "--bpo_top_labels", "1100",
    "--mfo_top_labels", "450",
    "--cco_top_labels", "300",

    "--save_pt",
    "--save_npy",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in p.stdout:
    print(line, end="")

rc = p.wait()
print("return code:", rc)

device=cuda
Selected labels per aspect -> BPO: 1100, MFO: 450, CCO: 300 (total=1850)
num_labels=1850
Discovered 20258 pdb files under /content/pdb_scratch

building sample index: 100%|██████████| 20258/20258 [00:00<00:00, 413249.41it/s]
label-matched pdbs: 20120
whitelist-matched pdbs: 20120
usable labeled pdbs: 20120
iterative stratification unavailable; using fallback split
train: n=16096 mean_labels=33.47
valid: n=2012 mean_labels=33.57
test: n=2012 mean_labels=33.59
Loaded encoder checkpoint. missing=0 unexpected=0
/content/micromamba/envs/torchdrug38/lib/python3.8/site-packages/torch/cuda/__init__.py:155: UserWarning: 
NVIDIA RTX PRO 6000 Blackwell Server Edition with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_70 sm_75 sm_80 sm_86.
If you want to use the NVIDIA RTX PRO 6000 Blackwell Server Edition GPU with PyTorch, please check the instructions at https://pytorch.org/g

# Train on GearNet pt (the embeds we prepared before)

In [ ]:
!python /content/scripts/train_go_mlp_cafa_from_saved_splits_i.py \
  --train_terms_tsv /content/train_terms.tsv \
  --go_obo /content/go-basic.obo \
  --ia_txt /content/IA.txt \
  --embed_dir /content/gearnet_embeds \
  --output_dir /content/go_mlp_from_saved_embeds_gearnet_all \
  --epochs 100 \
  --batch_size 5120 \
  #--use_cafa_callback

2026-04-03 13:17:02.020336: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-03 13:17:02.040307: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775222222.063131   69289 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775222222.070419   69289 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775222222.087959   69289 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!tar -cf go_mlp_from_saved_embeds_gearnet_all.tar go_mlp_from_saved_embeds_gearnet_all/

In [ ]:
!gsutil cp go_mlp_from_saved_embeds_gearnet_all.tar gs://protein_shards

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://go_mlp_from_saved_embeds_gearnet_all.tar [Content-Type=application/x-tar]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downlo

In [ ]:
!gsutil ls gs://protein_shards

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://protein_shards/IA.txt
gs://protein_shards/Schake_model_v2.py
gs://protein_shards/Schake_trained_weights.pt
gs://protein_shards/VarBatchSampler.py
gs://protein_shards/experimental_structures_valid_test.tar
gs://protein_shards/gearnet_embeds.zip
gs://protein_shards/gearnet_experimental_embeds.tar
gs://protein_shards/go-basic.obo
gs://protein_shards/go_mlp_from_saved_embeds_gearnet_all.tar
gs://protein_shards/t5_prot_baseline_train_embeds.npy.zip
gs://protein_shards/t5_prot_baseline_train_ids.npy.zip
gs://protein_shards/train_taxonomy.tsv
gs://protein_shards/train_terms.tsv
gs://protein_shards/pt_shards/
gs://protein_shards/shards/
gs://protein_shards/wheels/


# Train an ensemble with Cross Att Fusion of embeddings

In [ ]:
!python /content/scripts/train_go_mlp_cafa_bilinear_fusion.py \
  --train_terms_tsv /content/train_terms.tsv \
  --go_obo /content/go-basic.obo \
  --ia_txt /content/IA.txt \
  --split_embed_dir /content/gearnet_embeds \
  --other_ids_npy /content/train_ids.npy \
  --other_embeds_npy /content/train_embeds.npy \
  --output_dir /content/out_bilinear_fusion \
  --epochs 60 \
  --batch_size 1024 \
  --lr 2e-4 \
  --hidden_dim 512 \
  --bilinear_rank 128 \
  --dropout 0.20 \
  --modality_drop_prob 0.10

2026-04-02 16:32:13.756301: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775147533.778529   36292 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775147533.785846   36292 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775147533.802684   36292 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775147533.802720   36292 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775147533.802723   36292 computation_placer.cc:177] computation placer alr

In [ ]:
FINAL TEST
                ns     fmax    wfmax  best_tau
biological_process 0.430142 0.383805      0.11
molecular_function 0.648200 0.583704      0.18
cellular_component 0.689435 0.572120      0.21

# Download high-quality experimental pdb structure for the same test set

In [ ]:
!tar -cf experimental_structures_valid_test.tar experimental_structures_valid_test/

In [ ]:
!gsutil -m cp experimental_structures_valid_test.tar  gs://protein_shards

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://experimental_structures_valid_test.tar [Content-Type=application/x-tar]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables download

In [ ]:
!gsutil -m ls -l gs://protein_shards

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
   1068575  2026-04-02T15:27:20Z  gs://protein_shards/IA.txt
     44211  2026-02-27T08:07:44Z  gs://protein_shards/Schake_model_v2.py
    212619  2026-02-27T08:08:09Z  gs://protein_shards/Schake_trained_weights.pt
      6751  2026-02-27T08:08:00Z  gs://protein_shards/VarBatchSampler.py
1981962240  2026-04-03T12:20:01Z  gs://protein_shards/experimental_structures_valid_test.tar
2883149323  2026-04-02T15:41:35Z  gs://protein_shards/gearnet_embeds.zip
  30992530  2026-04-02T15:27:36Z  gs://protein_shards/go-basic.obo
 656247815  2026-04-02T15:25:56Z  gs://protein_shards/t5_prot_baseline_train_embeds.npy.zip
    965533  2026-04-02T15:27:03Z  gs://protein_shards/t5_prot_baseline_train_ids.npy.zip
   1835615  2026-02-27T08:09:47Z  gs://p

# Turn those pdbs into embeddings

In [ ]:
env = os.environ.copy()

# TorchDrug / compiled extensions
env["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"

# safe matplotlib backend
env["MPLBACKEND"] = "Agg"

# make sure micromamba env binaries are visible
env["PATH"] = "/content/micromamba/envs/torchdrug38/bin:" + env["PATH"]

cmd = [
    "/content/micromamba/envs/torchdrug38/bin/python",
    "-u",
    "/content/gearnet_embed_downloaded_experimental_pdbs.py",

    "--reference_embed_dir", "/content/gearnet_embeds",
    "--download_root", "/content/experimental_structures_valid_test",
    "--checkpoint", "/content/checkpoints/mc_gearnet_edge.pth",
    "--output_dir", "/content/gearnet_experimental_embeds",

    "--batch_size", "2", #16 for G4 GPU
    "--num_workers", "20",

    "--save_pt",
    "--save_npy",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in p.stdout:
    print(line, end="")

rc = p.wait()
print("return code:", rc)

Streaming output truncated to the last 5000 lines.
  warnings.warn("Unknown residue `%s`. Treat as glycine" % type)
/content/micromamba/envs/torchdrug38/lib/python3.8/site-packages/torchdrug/data/feature.py:42: UserWarning: Unknown value ` MG`
  warnings.warn("Unknown value `%s`" % x)
/content/micromamba/envs/torchdrug38/lib/python3.8/site-packages/torchdrug/data/protein.py:212: UserWarning: Unknown residue `ATP`. Treat as glycine
  warnings.warn("Unknown residue `%s`. Treat as glycine" % type)
/content/micromamba/envs/torchdrug38/lib/python3.8/site-packages/torchdrug/data/feature.py:42: UserWarning: Unknown value `ATP`
  warnings.warn("Unknown value `%s`" % x)
/content/micromamba/envs/torchdrug38/lib/python3.8/site-packages/torchdrug/data/protein.py:212: UserWarning: Unknown residue `K`. Treat as glycine
  warnings.warn("Unknown residue `%s`. Treat as glycine" % type)
/content/micromamba/envs/torchdrug38/lib/python3.8/site-packages/torchdrug/data/feature.py:42: UserWarning: Unknown va

In [ ]:
!tar -cf gearnet_experimental_embeds.tar gearnet_experimental_embeds/

In [ ]:
!gsutil cp gearnet_experimental_embeds.tar gs://protein_shards/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://gearnet_experimental_embeds.tar [Content-Type=application/x-tar]...
\
Operation completed over 1 objects/90.2 MiB.                                     


In [ ]:
!gsutil -m ls gs://protein_shards/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://protein_shards/IA.txt
gs://protein_shards/Schake_model_v2.py
gs://protein_shards/Schake_trained_weights.pt
gs://protein_shards/VarBatchSampler.py
gs://protein_shards/experimental_structures_valid_test.tar
gs://protein_shards/gearnet_embeds.zip
gs://protein_shards/gearnet_experimental_embeds.tar
gs://protein_shards/go-basic.obo
gs://protein_shards/t5_prot_baseline_train_embeds.npy.zip
gs://protein_shards/t5_prot_baseline_train_ids.npy.zip
gs://protein_shards/train_taxonomy.tsv
gs://protein_shards/train_terms.tsv
gs://protein_shards/pt_shards/
gs://protein_shards/shards/
gs://protein_shards/wheels/


In [ ]:
#Interesting how many were broken due to this seems like just 26
#   warnings.warn("Unknown residue `%s`. Treat as glycine" % type)
# /content/micromamba/envs/torchdrug38/lib/python3.8/site-packages/torchdrug/data/feature.py:42: UserWarning: Unknown value `TRS`
#   warnings.warn("Unknown value `%s`" % x)

# Run my trained on gearnet embeds model - on intersection of (concat of train valid) for downloaded pdbs and for real (train and valid)


In [ ]:
!python /content/scripts/eval_trained_go_mlp_on_original_vs_experimental_gearnet_i.py \
  --train_terms_tsv /content/train_terms.tsv \
  --go_obo /content/go-basic.obo \
  --ia_txt /content/IA.txt \
  --original_embed_dir /content/gearnet_embeds \
  --experimental_embed_dir /content/gearnet_experimental_embeds \
  --trained_model_path /content/go_mlp_from_saved_embeds_gearnet_all/best_model.keras \
  --output_dir /content/experimental_vs_original_eval \
  --batch_size 4096

2026-04-03 13:24:47.718405: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-03 13:24:47.738780: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775222687.761568   77132 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775222687.768973   77132 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775222687.786990   77132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
===== TEST =====
[test] common accessions: 1155
[test] original x shape: (1155, 3072)
[test] experimental x shape: (1155, 3072)
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 744ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step

[test] ORIGINAL AlphaDB GearNet
                ns     fmax    wfmax  best_tau
biological_process 0.423721 0.367449      0.08
molecular_function 0.600180 0.504413      0.10
cellular_component 0.631999 0.506875      0.10
mean_Fmax : 0.551967
mean_WFmax: 0.459579

[test] EXPERIMENTAL PDB GearNet
                ns     fmax    wfmax  best_tau
biological_process 0.355751 0.302191      0.03
molecular_function 0.550973 0.447880      0.05
cellular_component 0.583331 0.433478      0.06
mean_Fmax : 0.496685
mean_WFmax: 0.394516

[test] DELTA experimental - original
delta_Fmax : -0.055282
delta_WFmax: -0.065063

In [ ]:
Naive from GOMix poster
BPO 0.318
MFO 0.304
CCO 0.605

In [ ]:
#GearNet embeds on full validation set
FINAL TEST on ALL 12497 accessions
                ns     fmax    wfmax  best_tau
biological_process 0.409207 0.364938      0.10
molecular_function 0.638563 0.578775      0.15
cellular_component 0.663726 0.545321      0.16
mean_Fmax : 0.570499
mean_WFmax: 0.496344
Saved outputs to: /content/go_mlp_from_saved_embeds_gearnet_all

In [ ]:
#T5 baseline STRONG from LLM embeds from kaggle
FINAL TEST on All 12497 accessions
                ns     fmax    wfmax  best_tau
biological_process 0.432546 0.387054      0.09
molecular_function 0.650834 0.587200      0.16
cellular_component 0.688379 0.574110      0.19
mean_Fmax : 0.590587
mean_WFmax: 0.516121

In [ ]:
# !tar -cf go_mlp_from_saved_embeds_gearnet_all.tar go_mlp_from_saved_embeds_gearnet_all/

In [ ]:
# !gsutil cp go_mlp_from_saved_embeds_gearnet_all.tar gs://protein_shards

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://go_mlp_from_saved_embeds_gearnet_all.tar [Content-Type=application/x-tar]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downlo

In [ ]:
!gsutil ls gs://protein_shards

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://protein_shards/IA.txt
gs://protein_shards/Schake_model_v2.py
gs://protein_shards/Schake_trained_weights.pt
gs://protein_shards/VarBatchSampler.py
gs://protein_shards/experimental_structures_valid_test.tar
gs://protein_shards/gearnet_embeds.zip
gs://protein_shards/gearnet_experimental_embeds.tar
gs://protein_shards/go-basic.obo
gs://protein_shards/go_mlp_from_saved_embeds_gearnet_all.tar
gs://protein_shards/t5_prot_baseline_train_embeds.npy.zip
gs://protein_shards/t5_prot_baseline_train_ids.npy.zip
gs://protein_shards/train_taxonomy.tsv
gs://protein_shards/train_terms.tsv
gs://protein_shards/pt_shards/
gs://protein_shards/shards/
gs://protein_shards/wheels/


# 🤩 Lets do some amazing error analysis and visualization

After we ran blocks:


In [ ]:
!gsutil ls gs://protein_shards

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://protein_shards/IA.txt
gs://protein_shards/Schake_model_v2.py
gs://protein_shards/Schake_trained_weights.pt
gs://protein_shards/VarBatchSampler.py
gs://protein_shards/experimental_structures_valid_test.tar
gs://protein_shards/gearnet_embeds.zip
gs://protein_shards/gearnet_experimental_embeds.tar
gs://protein_shards/go-basic.obo
gs://protein_shards/go_mlp_from_saved_embeds_gearnet_all.tar
gs://protein_shards/t5_prot_baseline_train_embeds.npy.zip
gs://protein_shards/t5_prot_baseline_train_ids.npy.zip
gs://protein_shards/train_taxonomy.tsv
gs://protein_shards/train_terms.tsv
gs://protein_shards/pt_shards/
gs://protein_shards/shards/
gs://protein_shards/wheels/


In [ ]:
#untar go_mlp_from_saved_embeds_gearnet_all.tar

In [ ]:
#take gearnet embeds and gearnet experimental embeds
#use mlp to evaluate on both

In [ ]:
#pull trained model tested on all 12497
#say which failed and how bad
#tell pdb names

In [ ]:
#pull

In [ ]:
!ls

In [ ]:
#USE embeds to try similarity search and determine function by similarity

In [ ]:
#Find most similar and most dissimilar exp and from 1500

In [ ]:
#OVERALL WE VISUALIZE
#THE WORST PREDICTIONS FOR
#THE MOST SIMILAR AND DISSIMILAR 5 or 3
#IS IT SAME ONES THAT GOT DIFFERENT PREDS

# Lets try ESM-GearNet


Lets first try on several embeds and then do a similar format as before embed all script

In [ ]:
!apt-get -y install aria2

!mkdir -p /content/checkpoints /content/protein-model-weights/esm-model-weights

# fast Zenodo download for the fusion checkpoint
!aria2c -c -x 16 -s 16 -k 1M \
  -d /content/checkpoints \
  -o mc_esm_gearnet.pth \
  "https://zenodo.org/records/10034578/files/mc_esm_gearnet.pth?download=1"

# fast direct download for ESM-2 650M weights
!aria2c -c -x 16 -s 16 -k 1M \
  -d /content/protein-model-weights/esm-model-weights \
  -o esm2_t33_650M_UR50D.pt \
  "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libc-ares2
The following NEW packages will be installed:
  aria2 libaria2-0 libc-ares2
0 upgraded, 3 newly installed, 0 to remove and 72 not upgraded.
Need to get 1,513 kB of archives.
After this operation, 5,441 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libc-ares2 amd64 1.18.1-1ubuntu0.22.04.3 [45.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libaria2-0 amd64 1.36.0-1 [1,086 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 aria2 amd64 1.36.0-1 [381 kB]
Fetched 1,513 kB in 1s (1,365 kB/s)
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubun

In [ ]:
env = os.environ.copy()
env["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"
env["MPLBACKEND"] = "Agg"
env["PATH"] = "/content/micromamba/envs/torchdrug38/bin:" + env["PATH"]

cmd = [
    "/content/micromamba/envs/torchdrug38/bin/python",
    "-u",
    "/content/esm_gearnet_unified_embedding_pipeline.py",

    "--mode", "all",
    "--pdb_root", "/content/pdb_scratch",
    "--train_terms_tsv", "/content/train_terms.tsv",
    "--train_taxonomy_tsv", "/content/train_taxonomy.tsv",
    "--checkpoint", "/content/checkpoints/mc_esm_gearnet.pth",
    "--esm_weight_dir", "/content/protein-model-weights/esm-model-weights",
    "--output_dir", "/content/esm_gearnet_all_pdbs_embeds",

    "--batch_size", "64",
    "--num_workers", "32",
    "--max_length", "550",

    "--bpo_top_labels", "1100",
    "--mfo_top_labels", "450",
    "--cco_top_labels", "300",

    "--save_pt",
    "--save_npy",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in p.stdout:
    print(line, end="")

rc = p.wait()
print("return code:", rc)

Selected labels per aspect -> BPO: 1100, MFO: 450, CCO: 300 (total=1850)
num_labels=1850
Discovered 127832 pdb files under /content/pdb_scratch

building sample index: 100%|██████████| 127832/127832 [00:00<00:00, 410324.94it/s]
label-matched pdbs: 127053
whitelist-matched pdbs: 127053
usable labeled pdbs: 127053
iterative stratification unavailable; using fallback split
wrote /content/esm_gearnet_all_pdbs_embeds/splits.csv
train: n=101643 mean_labels=32.35
valid: n=12705 mean_labels=32.37
test: n=12705 mean_labels=32.37
device=cuda
/content/micromamba/envs/torchdrug38/lib/python3.10/site-packages/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(
Loaded ESM-GearNet checkpoint. missing=0 unexpected=0

embed:train:   0%|          | 0/1589 [00:00<?, ?it/s][13:01:13] Explicit valence for atom # 73 C, 5, is greater than permitted
[13:01:17] Explicit valence for atom # 1048 O, 3, is greater than permitted
[

In [ ]:
!gsutil -m cp gs://protein_shards/experimental_structures_valid_test.tar /content/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://protein_shards/experimental_structures_valid_test.tar...
- [1/1 files][  1.8 GiB/  1.8 GiB] 100% Done                                    
Operation completed over 1 objects/1.8 GiB.                                      


In [ ]:
!rm -rf experimental_structures_valid_test/

In [ ]:
!mkdir -p /content/experimental_structures_valid_test
!tar -xvf /content/experimental_structures_valid_test.tar -C /content/

experimental_structures_valid_test/
experimental_structures_valid_test/summary.json
experimental_structures_valid_test/test/
experimental_structures_valid_test/test/test_experimental_structures.csv
experimental_structures_valid_test/test/structures/
experimental_structures_valid_test/test/structures/2CMW.pdb
experimental_structures_valid_test/test/structures/3TUF.pdb
experimental_structures_valid_test/test/structures/7U4C.pdb
experimental_structures_valid_test/test/structures/5YQI.pdb
experimental_structures_valid_test/test/structures/4DY0.pdb
experimental_structures_valid_test/test/structures/1SZ7.pdb
experimental_structures_valid_test/test/structures/7QCR.pdb
experimental_structures_valid_test/test/structures/2PXX.pdb
experimental_structures_valid_test/test/structures/5B8D.pdb
experimental_structures_valid_test/test/structures/8J0Q.pdb
experimental_structures_valid_test/test/structures/8XV6.pdb
experimental_structures_valid_test/test/structures/2P52.pdb
experimental_structures_valid_

In [ ]:
import os

env = os.environ.copy()
env["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"
env["MPLBACKEND"] = "Agg"
env["PATH"] = "/content/micromamba/envs/torchdrug38/bin:" + env["PATH"]

cmd = [
    "/content/micromamba/envs/torchdrug38/bin/python",
    "-u",
    "/content/esm_gearnet_unified_embedding_pipeline.py",

    "--mode", "experimental",
    "--reference_embed_dir", "/content/esm_gearnet_all_pdbs_embeds",
    "--download_root", "/content/experimental_structures_valid_test",
    "--checkpoint", "/content/checkpoints/mc_esm_gearnet.pth",
    "--esm_weight_dir", "/content/protein-model-weights/esm-model-weights",
    "--output_dir", "/content/esm_gearnet_experimental_embeds",

    "--batch_size", "8",
    "--num_workers", "64",
    "--max_length", "550",

    "--save_pt",
    "--save_npy",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in p.stdout:
    print(line, end="")

rc = p.wait()
print("return code:", rc)

We move the tars to the gcloud

In [ ]:
!tar -cvf /content/esm_gearnet_all_pdbs_embeds.tar -C /content esm_gearnet_all_pdbs_embeds

esm_gearnet_all_pdbs_embeds/
esm_gearnet_all_pdbs_embeds/valid_embeddings.pt
esm_gearnet_all_pdbs_embeds/train_embeddings.npy
esm_gearnet_all_pdbs_embeds/test_embeddings.pt
esm_gearnet_all_pdbs_embeds/train_pdb_files.npy
esm_gearnet_all_pdbs_embeds/valid_pdb_files.npy
esm_gearnet_all_pdbs_embeds/test_pdb_files.npy
esm_gearnet_all_pdbs_embeds/test_index.csv
esm_gearnet_all_pdbs_embeds/valid_targets.npy
esm_gearnet_all_pdbs_embeds/valid_index.csv
esm_gearnet_all_pdbs_embeds/test_targets.npy
esm_gearnet_all_pdbs_embeds/label_vocab.json
esm_gearnet_all_pdbs_embeds/train_accessions.npy
esm_gearnet_all_pdbs_embeds/train_targets.npy
esm_gearnet_all_pdbs_embeds/splits.csv
esm_gearnet_all_pdbs_embeds/train_embeddings.pt
esm_gearnet_all_pdbs_embeds/train_index.csv
esm_gearnet_all_pdbs_embeds/test_embeddings.npy
esm_gearnet_all_pdbs_embeds/valid_embeddings.npy
esm_gearnet_all_pdbs_embeds/test_accessions.npy
esm_gearnet_all_pdbs_embeds/valid_accessions.npy


In [ ]:
!tar -cvf /content/esm_gearnet_experimental_embeds.tar -C /content esm_gearnet_experimental_embeds

esm_gearnet_experimental_embeds/
esm_gearnet_experimental_embeds/valid_embeddings.pt
esm_gearnet_experimental_embeds/test_embeddings.pt
esm_gearnet_experimental_embeds/valid_pdb_files.npy
esm_gearnet_experimental_embeds/test_pdb_files.npy
esm_gearnet_experimental_embeds/test_index.csv
esm_gearnet_experimental_embeds/valid_targets.npy
esm_gearnet_experimental_embeds/valid_index.csv
esm_gearnet_experimental_embeds/test_targets.npy
esm_gearnet_experimental_embeds/label_vocab.json
esm_gearnet_experimental_embeds/test_bad_samples.csv
esm_gearnet_experimental_embeds/valid_bad_samples.csv
esm_gearnet_experimental_embeds/test_embeddings.npy
esm_gearnet_experimental_embeds/valid_embeddings.npy
esm_gearnet_experimental_embeds/test_accessions.npy
esm_gearnet_experimental_embeds/valid_accessions.npy


In [ ]:
!gsutil -m cp /content/esm_gearnet_all_pdbs_embeds.tar gs://protein_shards/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file:///content/esm_gearnet_all_pdbs_embeds.tar [Content-Type=application/x-tar]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downlo

In [ ]:
!gsutil -m cp /content/esm_gearnet_experimental_embeds.tar gs://protein_shards/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file:///content/esm_gearnet_experimental_embeds.tar [Content-Type=application/x-tar]...
- [1/1 files][ 79.3 MiB/ 79.3 MiB] 100% Done                                    
Operation completed over 1 objects/79.3 MiB.                                     


# Lets train on all pdbs with ESM-Gearnet embeds

In [ ]:
!gsutil -m ls gs://protein_shards/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://protein_shards/IA.txt
gs://protein_shards/Schake_model_v2.py
gs://protein_shards/Schake_trained_weights.pt
gs://protein_shards/VarBatchSampler.py
gs://protein_shards/esm_gearnet_all_pdbs_embeds.tar
gs://protein_shards/esm_gearnet_experimental_embeds.tar
gs://protein_shards/experimental_structures_valid_test.tar
gs://protein_shards/gearnet_embeds.zip
gs://protein_shards/gearnet_experimental_embeds.tar
gs://protein_shards/go-basic.obo
gs://protein_shards/go_mlp_from_saved_embeds_gearnet_all.tar
gs://protein_shards/t5_prot_baseline_train_embeds.npy.zip
gs://protein_shards/t5_prot_baseline_train_ids.npy.zip
gs://protein_shards/train_taxonomy.tsv
gs://protein_shards/train_terms.tsv
gs://protein_shards/pt_shards/
gs://protein_shards

In [ ]:
!gsutil -m cp gs://protein_shards/esm_gearnet_all_pdbs_embeds.tar /content/
!gsutil -m cp gs://protein_shards/esm_gearnet_experimental_embeds.tar /content/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://protein_shards/esm_gearnet_all_pdbs_embeds.tar...
| [1/1 files][  5.5 GiB/  5.5 GiB] 100% Done  82.6 MiB/s ETA 00:00:00           
Operation completed over 1 objects/5.5 GiB.                                      
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://protein_shards/esm_gearnet_experimental_embeds.tar...
- [1/1 files][ 79.3 MiB/ 79.3 MiB] 100% Done                                    
Operation completed over 1 objects/79.3 MiB.                                     


In [ ]:
!tar -xvf /content/esm_gearnet_all_pdbs_embeds.tar -C /content/
!tar -xvf /content/esm_gearnet_experimental_embeds.tar -C /content/

esm_gearnet_all_pdbs_embeds/
esm_gearnet_all_pdbs_embeds/valid_embeddings.pt
esm_gearnet_all_pdbs_embeds/train_embeddings.npy
esm_gearnet_all_pdbs_embeds/test_embeddings.pt
esm_gearnet_all_pdbs_embeds/train_pdb_files.npy
esm_gearnet_all_pdbs_embeds/valid_pdb_files.npy
esm_gearnet_all_pdbs_embeds/test_pdb_files.npy
esm_gearnet_all_pdbs_embeds/test_index.csv
esm_gearnet_all_pdbs_embeds/valid_targets.npy
esm_gearnet_all_pdbs_embeds/valid_index.csv
esm_gearnet_all_pdbs_embeds/test_targets.npy
esm_gearnet_all_pdbs_embeds/label_vocab.json
esm_gearnet_all_pdbs_embeds/train_accessions.npy
esm_gearnet_all_pdbs_embeds/train_targets.npy
esm_gearnet_all_pdbs_embeds/splits.csv
esm_gearnet_all_pdbs_embeds/train_embeddings.pt
esm_gearnet_all_pdbs_embeds/train_index.csv
esm_gearnet_all_pdbs_embeds/test_embeddings.npy
esm_gearnet_all_pdbs_embeds/valid_embeddings.npy
esm_gearnet_all_pdbs_embeds/test_accessions.npy
esm_gearnet_all_pdbs_embeds/valid_accessions.npy
esm_gearnet_experimental_embeds/
esm_gear

In [ ]:
!python /content/scripts/train_go_mlp_cafa_from_saved_splits_I.py \
  --train_terms_tsv /content/train_terms.tsv \
  --go_obo /content/go-basic.obo \
  --ia_txt /content/IA.txt \
  --embed_dir /content/esm_gearnet_all_pdbs_embeds \
  --output_dir /content/go_mlp_from_saved_embeds_gearnet_all \
  --epochs 100 \
  --batch_size 5120 \
  #--use_cafa_callback

2026-04-11 16:08:49.752570: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-11 16:08:50.164958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775923730.350465   14705 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775923730.396609   14705 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775923730.769013   14705 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

# ESM-Gearnet structure swapping experiment

In [ ]:
!python /content/scripts/eval_trained_go_mlp_on_original_vs_experimental_gearnet.py \
  --train_terms_tsv /content/train_terms.tsv \
  --go_obo /content/go-basic.obo \
  --ia_txt /content/IA.txt \
  --original_embed_dir /content/esm_gearnet_all_pdbs_embeds \
  --experimental_embed_dir /content/esm_gearnet_experimental_embeds \
  --trained_model_path /content/go_mlp_from_saved_embeds_gearnet_all/best_model.keras \
  --output_dir /content/esm_gearnet_experimental_vs_original_eval \
  --batch_size 4096

2026-04-11 16:18:48.578244: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-11 16:18:48.599140: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775924328.621072   23147 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775924328.628372   23147 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775924328.645979   23147 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

# Baseline naive frequency with CAFA evaluation

In [ ]:
!python /content/scripts/eval_go_naive_frequency_baseline.py \
  --train_terms_tsv /content/train_terms.tsv \
  --go_obo /content/go-basic.obo \
  --ia_txt /content/IA.txt \
  --embed_dir /content/esm_gearnet_all_pdbs_embeds \
  --output_dir /content/go_naive_frequency_baseline_esm_gearnet

2026-04-11 16:24:33.673240: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-11 16:24:33.694356: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775924673.716755   24871 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775924673.724068   24871 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775924673.742347   24871 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

# Some drafts

In [ ]:
!python /content/scripts/train_go_mlp_cafa_from_saved_splits.py \
  --train_terms_tsv /content/train_terms.tsv \
  --go_obo /content/go-basic.obo \
  --ia_txt /content/IA.txt \
  --embed_dir /content/go_finetune_runs \
  --output_dir /content/go_mlp_from_saved_embeds_residual \
  --epochs 60 \
  --batch_size 5120 \
  --lr 3e-3 \
  --weight_decay 1e-4 \
  --early_stop_patience 6

2026-04-02 09:41:11.335637: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-02 09:41:11.357106: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775122871.379389   25767 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775122871.386618   25767 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775122871.404515   25767 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!pip install -U xgboost scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 96.9 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
!python /content/scripts/train_go_xgb_cafa_from_saved_splits.py \
  --train_terms_tsv /content/train_terms.tsv \
  --go_obo /content/go-basic.obo \
  --ia_txt /content/IA.txt \
  --embed_dir /content/go_finetune_runs \
  --output_dir /content/go_xgb_from_saved_embeds \
  --n_estimators 500 \
  --max_depth 6 \
  --learning_rate 0.05 \
  --subsample 0.8 \
  --colsample_bytree 0.8 \
  --reg_lambda 2.0 \
  --tree_method hist

train (99627, 3072) (99627, 1850)
valid (12369, 3072) (12369, 1850)
test  (12497, 3072) (12497, 1850)
aspect counts:
BPO    1100
MFO     450
CCO     300

VALID
                ns  fmax  wfmax  best_tau
biological_process   0.0    0.0       0.5
molecular_function   0.0    0.0       0.5
cellular_component   0.0    0.0       0.5
mean_Fmax : 0.0
mean_WFmax: 0.0

FINAL TEST
                ns  fmax  wfmax  best_tau
biological_process   0.0    0.0       0.5
molecular_function   0.0    0.0       0.5
cellular_component   0.0    0.0       0.5
mean_Fmax : 0.0
mean_WFmax: 0.0
Saved outputs to: /content/go_xgb_from_saved_embeds


In [ ]:
!python /content/scripts/train_go_mlp_3heads_cafa_from_saved_splits.py \
  --train_terms_tsv /content/train_terms.tsv \
  --go_obo /content/go-basic.obo \
  --ia_txt /content/IA.txt \
  --embed_dir /content/go_finetune_runs \
  --output_dir /content/go_mlp_3heads_from_saved_embeds \
  --epochs 60 \
  --batch_size 4096 \
  --lr 3e-4 \
  --weight_decay 1e-4 \
  --early_stop_patience 6 \
  --use_cafa_callback

2026-04-02 10:04:52.732312: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-02 10:04:52.752116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775124292.774413   35363 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775124292.781612   35363 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775124292.799458   35363 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
env = os.environ.copy()

# safest for TF on your setup
env["CUDA_VISIBLE_DEVICES"] = "-1"

cmd = [
    "python",
    "-u",
    "/content/train_go_mlp_3heads_cafa_from_saved_splits.py",
    "--train_terms_tsv", "/content/train_terms.tsv",
    "--go_obo", "/content/go-basic.obo",
    "--ia_txt", "/content/IA.txt",
    "--embed_dir", "/content/go_finetune_runs/run1",
    "--output_dir", "/content/go_mlp_3heads_from_saved_embeds",
    "--epochs", "60",
    "--batch_size", "4096",
    "--lr", "3e-4",
    "--weight_decay", "1e-4",
    "--early_stop_patience", "6",
    "--use_cafa_callback",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in p.stdout:
    print(line, end="")

rc = p.wait()
print("return code:", rc)

2026-04-02 10:02:44.323957: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-02 10:02:44.344151: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775124164.366596   34746 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775124164.374001   34746 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775124164.391993   34746 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
env = os.environ.copy()

# REQUIRED for TorchDrug JIT extension
env["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"

# ensure micromamba env binaries (ninja etc.) are visible
env["PATH"] = "/content/micromamba/envs/torchdrug38/bin:" + env["PATH"]

cmd = [
    "/content/micromamba/envs/torchdrug38/bin/python",
    "-u",
    "/content/gearnet_go_finetune_pipeline.py",

    "--pdb_root", "/content/pdb_scratch",
    "--train_terms_tsv", "/content/train_terms.tsv",
    "--train_taxonomy_tsv", "/content/train_taxonomy.tsv",
    "--checkpoint", "/content/checkpoints/mc_gearnet_edge.pth",
    "--output_dir", "/content/go_finetune_runs/run1",

    "--num_labels", "1500",
    "--batch_size", "2",
    "--epochs", "10",
    "--lr_encoder", "1e-5",
    "--lr_head", "1e-3",
    "--num_workers", "10",
    "--freeze_encoder_epochs", "10",
]

p = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in p.stdout:
    print(line, end="")  # real-time streaming

rc = p.wait()
print("return code:", rc)

device=cuda
Discovered 6416 pdb files under /content/pdb_scratch

building sample index: 100%|██████████| 6416/6416 [00:00<00:00, 184112.74it/s]
label-matched pdbs: 6375
whitelist-matched pdbs: 6375
usable labeled pdbs: 6375
iterative stratification unavailable; using fallback split
train: n=5099 mean_labels=32.53
valid: n=638 mean_labels=32.76
test: n=638 mean_labels=32.82
Loaded encoder checkpoint. missing=0 unexpected=0

train:epoch1:   2%|▏         | 49/2549 [00:10<05:19,  7.81it/s]train epoch=1 step=50/2549 loss=2.1824

train:epoch1:   4%|▍         | 100/2549 [00:18<03:40, 11.09it/s]train epoch=1 step=100/2549 loss=2.2923

train:epoch1:   6%|▌         | 150/2549 [00:26<05:22,  7.43it/s]train epoch=1 step=150/2549 loss=1.4968

train:epoch1:   8%|▊         | 199/2549 [00:36<04:09,  9.43it/s]train epoch=1 step=200/2549 loss=0.4141

train:epoch1:  10%|▉         | 249/2549 [00:47<06:08,  6.25it/s]train epoch=1 step=250/2549 loss=1.0559

train:epoch1:  12%|█▏        | 300/2549 [00:56<07

KeyboardInterrupt: 

In [ ]:
PDB_ROOT = "/content/pdb_scratch"
TRAIN_TERMS_PATH = "/content/train_terms.tsv"
TAXONOMY_PATH = "/content/train_taxonomy.tsv"

pdb_files = sorted(glob.glob(os.path.join(PDB_ROOT, "**", "*.pdb"), recursive=True))
print("num pdb files:", len(pdb_files))
print("first 5 pdb basenames:", [os.path.basename(x) for x in pdb_files[:5]])

train_terms = pd.read_csv(TRAIN_TERMS_PATH, sep="\t")
print("train_terms columns:", train_terms.columns.tolist())
print("first 10 EntryID:", train_terms["EntryID"].astype(str).head(10).tolist())

if os.path.exists(TAXONOMY_PATH):
    tax = pd.read_csv(TAXONOMY_PATH, sep="\t")
    print("taxonomy columns:", tax.columns.tolist())
    if "EntryID" in tax.columns:
        print("first 10 taxonomy EntryID:", tax["EntryID"].astype(str).head(10).tolist())

num pdb files: 6416
first 5 pdb basenames: ['AF-0000000000249450.pdb', 'AF-0000000000261410.pdb', 'AF-0000000065714493.pdb', 'AF-0000000065721713.pdb', 'AF-0000000065723180.pdb']
train_terms columns: ['EntryID', 'term', 'aspect']
first 10 EntryID: ['A0A009IHW8', 'A0A009IHW8', 'A0A009IHW8', 'A0A009IHW8', 'A0A009IHW8', 'A0A009IHW8', 'A0A009IHW8', 'A0A009IHW8', 'A0A009IHW8', 'A0A009IHW8']
taxonomy columns: ['EntryID', 'taxonomyID']
first 10 taxonomy EntryID: ['Q8IXT2', 'Q04418', 'A8DYA3', 'Q9UUI3', 'Q57ZS4', 'P17571', 'Q9JMA2', 'Q8R2Z3', 'Q8IZR5', 'P04014']


In [ ]:
PDB_ROOT = "/content/pdb_scratch"
TRAIN_TERMS_PATH = "/content/train_terms.tsv"

def extract_uniprot_from_filename(path):
    base = os.path.basename(path)
    stem = os.path.splitext(base)[0]

    # AF-A0A0S2Z5D6-F1 → A0A0S2Z5D6
    if stem.startswith("AF-"):
        parts = stem.split("-")
        if len(parts) >= 2:
            return parts[1]

    return None


# load data
pdb_files = sorted(glob.glob(os.path.join(PDB_ROOT, "**", "*.pdb"), recursive=True))
entry_ids = set(pd.read_csv(TRAIN_TERMS_PATH, sep="\t")["EntryID"].astype(str))

# test overlap
matched = 0
examples = []

for p in pdb_files:
    uid = extract_uniprot_from_filename(p)
    if uid and uid in entry_ids:
        matched += 1
        if len(examples) < 5:
            examples.append((os.path.basename(p), uid))

print("total pdb:", len(pdb_files))
print("matched pdb:", matched)
print("coverage %:", matched / len(pdb_files) * 100)

print("\nexamples:")
for x in examples:
    print(x)

total pdb: 6416
matched pdb: 6383
coverage %: 99.4856608478803

examples:
('AF-A0A021WW32-F1.pdb', 'A0A021WW32')
('AF-A0A023G9N9-F1.pdb', 'A0A023G9N9')
('AF-A0A023T787-F1.pdb', 'A0A023T787')
('AF-A0A024QZT0-F1.pdb', 'A0A024QZT0')
('AF-A0A024R5J5-F1.pdb', 'A0A024R5J5')


In [ ]:
TAR_PATH = sorted(glob.glob("/content/shards/*.tar"))[0]
EXTRACT_DIR = "/content/pdb_scratch/shard0"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with tarfile.open(TAR_PATH, "r") as tar:
    members = [m for m in tar if m.isfile() and m.name.endswith(".pdb")]
    tar.extractall(EXTRACT_DIR, members=members)

pdb_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, "**", "*.pdb"), recursive=True))
print("num pdb files:", len(pdb_files))
print("first:", pdb_files[:3])

/tmp/ipykernel_50645/2499107815.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(EXTRACT_DIR, members=members)


num pdb files: 6416
first: ['/content/pdb_scratch/shard0/AF-0000000000249450.pdb', '/content/pdb_scratch/shard0/AF-0000000000261410.pdb', '/content/pdb_scratch/shard0/AF-0000000065714493.pdb']


In [ ]:
!ps

    PID TTY          TIME CMD
      1 ?        00:00:00 docker-init
      7 ?        00:00:13 node
      9 ?        00:00:04 oom_monitor.sh
     11 ?        00:00:00 run.sh
     12 ?        00:00:02 kernel_manager_
     42 ?        00:00:00 tail
     43 ?        00:00:00 tail
     81 ?        00:00:18 python3 <defunct>
     82 ?        00:00:04 colab-fileshim.
    133 ?        00:00:07 jupyter-server
    134 ?        00:00:00 dap_multiplexer
  15225 ?        00:00:23 python3
  33479 ?        00:00:01 language_servic
  33490 ?        00:01:57 node
  42693 ?        00:00:00 sleep
  42694 ?        00:00:00 ps


# Not working drafts

In [ ]:
!pip install graphein

In [ ]:
!ls

AF-O15552-F1-model_v6.pdb  Schake_model_v2.py	      train_terms.tsv
gcs_key.json		   Schake_trained_weights.pt  VarBatchSampler.py
pt_shards		   shards
sample_data		   train_taxonomy.tsv


In [ ]:
import inspect
from graphein.protein.graphs import construct_graph
print(construct_graph)
print(inspect.signature(construct_graph))

<function construct_graph at 0x7d339aef3d80>
(config: 'Optional[ProteinGraphConfig]' = None, name: 'Optional[str]' = None, path: 'Optional[Union[str, os.PathLike]]' = None, uniprot_id: 'Optional[str]' = None, pdb_code: 'Optional[str]' = None, df: 'Optional[pd.DataFrame]' = None, chain_selection: 'Union[str, List[str]]' = 'all', model_index: 'int' = 1, df_processing_funcs: 'Optional[List[Callable]]' = None, edge_construction_funcs: 'Optional[List[Callable]]' = None, edge_annotation_funcs: 'Optional[List[Callable]]' = None, node_annotation_funcs: 'Optional[List[Callable]]' = None, graph_annotation_funcs: 'Optional[List[Callable]]' = None, verbose: 'bool' = True) -> 'nx.Graph'


In [ ]:
from functools import partial

from graphein.protein.config import ProteinGraphConfig
from graphein.protein.edges.distance import (
    add_peptide_bonds,
    add_k_nn_edges,
    add_distance_threshold,
)
from graphein.protein.graphs import construct_graph
from graphein.ml.conversion import GraphFormatConvertor

config = ProteinGraphConfig(
    granularity="CA",
    edge_construction_functions=[
        add_peptide_bonds,
        partial(add_k_nn_edges, k=16, long_interaction_threshold=3),
        partial(add_distance_threshold, threshold=10.0, long_interaction_threshold=5),
    ],
)

g = construct_graph(config=config, path="AF-O15552-F1-model_v6.pdb")
to_pyg = GraphFormatConvertor(src_format="nx", dst_format="pyg")
data = to_pyg(g)

Output()

In [ ]:
data

Data(edge_index=[2, 5310], node_id=[330], coords=[330, 3], name='AF-O15552-F1-model_v6', num_nodes=330)

In [ ]:
import tempfile
import webdataset as wds
from torch.utils.data import Dataset
from torchdrug import data

KEY_RE = re.compile(r"AF-([A-Za-z0-9]+)(?:-F1)?$", re.IGNORECASE)

def entryid_from_key(key: str) -> str:
    k = key.strip().lstrip("./")
    k = os.path.basename(k)
    m = KEY_RE.search(k)
    if not m:
        raise ValueError(f"Cannot parse EntryID from key: {key}")
    return m.group(1)

def keep_only_pdb(sample: dict) -> bool:
    return "pdb" in sample

class TarPDBTorchDrugDataset(Dataset):
    def __init__(self, shard_paths, residue_feature="default", tmp_dir=None, verbose=False):
        self.samples = []
        self.residue_feature = residue_feature
        self.tmp_dir = tmp_dir
        self.verbose = verbose

        for shard_path in shard_paths:
            ds = wds.WebDataset([shard_path], shardshuffle=False).select(keep_only_pdb)
            for s in ds:
                key = s.get("__key__", "")
                if "pdb" in s:
                    self.samples.append({
                        "shard": shard_path,
                        "key": key,
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        meta = self.samples[idx]
        shard_path = meta["shard"]
        target_key = meta["key"]

        ds = wds.WebDataset([shard_path], shardshuffle=False).select(keep_only_pdb)
        for s in ds:
            if s.get("__key__", "") != target_key:
                continue

            entry_id = entryid_from_key(target_key)
            pdb_bytes = s["pdb"]

            fd, pdb_path = tempfile.mkstemp(suffix=".pdb", dir=self.tmp_dir)
            try:
                with os.fdopen(fd, "wb") as f:
                    f.write(pdb_bytes)

                protein = data.Protein.from_pdb(
                    pdb_path,
                    residue_feature=self.residue_feature
                )
            finally:
                try:
                    os.remove(pdb_path)
                except Exception:
                    pass

            return {"graph": protein, "id": entry_id}

        raise KeyError(f"Sample {target_key} not found in shard {shard_path}")